# Quantile regression: Gurobi original-LP algorithm comparison

Run all cells to print the final comparison. The data, T, projected dual-simplex solve,
and Dykstra retrieval are generated/executed **once**. Only the original LP's
`Method` and (for barrier without crossover) `Crossover` change. Each original
solve uses a fresh model and no warm start. Five repetitions rotate the order
of the five algorithms; every algorithm appears in each position once.
All runs are sequential. Times and objectives are arithmetic means over optimal
runs; violations are the largest observed across those runs. Failed runs are reported.

The helper functions below are copied unchanged from
`Liberti_QuantileRegression_m5000_RP_Retrieval.ipynb`. This notebook is standalone.
The original notebook is preserved.

The main ratio is `fixed projected pipeline wall time / mean original LP total wall time`.
Lower ratios mean a larger speed advantage for the projected workflow. A slower
original solve lowers this ratio; rank direct algorithms by their original solve
time. The numerator is one measurement, so repetitions estimate only variation
in the denominator.

Liberti et al., Table 5, m=5000, epsilon=0.2 reports 0.19. The user-observed ratio
is about 0.334. The paper uses CPLEX for the original LP, HiGHS for projected QR,
five random-projection runs, and a different machine and timing scope (including
instance reading/formulation work). Here shared D/b/A generation is excluded,
and timing uses `perf_counter()` wall seconds, matching the reference convention.
This experiment isolates the original Gurobi algorithm on this machine; it cannot
separately attribute CPLEX/HiGHS or hardware effects or establish exact reproduction.

References: local `Liberti et al.pdf`, Sections 3.2–3.8 and Table 5;
[Gurobi Method/Crossover documentation](https://docs.gurobi.com/projects/optimizer/en/current/reference/parameters.html).
Automatic is left as `Method=-1`; it may use concurrent LP optimization.
Barrier default crossover is `Method=2, Crossover=-1`; disabled crossover is
`Method=2, Crossover=0`. All other solver defaults are retained, including presolve
and tolerances. `THREADS=0` retains automatic thread allocation from the reference;
change it only as a separate controlled experiment.

Solver logs are quiet by default (`OUTPUT_FLAG=0`); use 1 to inspect them.
The projected pipeline is measured once, so only original-LP timings are averaged.
Detailed per-run measurements remain in `results`.


In [ ]:
import time
import platform
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.sparse.linalg import lsmr
import gurobipy as gp
from gurobipy import GRB

# Reference instance and projected/retrieval settings.
M = 5000
P = 399
DENSITY = 0.8
TAU = 0.2
EPSILON = 0.2
DATA_SEED = 42
PROJECTION_SEED = 42
MAX_ITER = 30
DELTA = 0.01

REPEATS = 5  # Five fresh original-LP solves per algorithm; arithmetic mean timings.
THREADS = 0  # Gurobi automatic allocation, as in the reference.
SOLVER_SEED = 0  # Gurobi default; fixed for all models.
OUTPUT_FLAG = 0  # Quiet output; set to 1 for Gurobi logs (may affect wall time).

PAPER_RATIO = 0.19
OBSERVED_RATIO = 0.334
ALGORITHMS = [
    ("Automatic", -1, -1),
    ("Primal simplex", 0, -1),
    ("Dual simplex", 1, -1),
    ("Barrier default crossover", 2, -1),
    ("Barrier without crossover", 2, 0),
]
assert REPEATS >= 1


In [ ]:
def generate_quantile_instance(
    m: int,
    p: int,
    density: float = 0.8,
    seed: int | None = None,
):
    """
    Generate one synthetic Quantile Regression instance.

    Parameters
    ----------
    m : int
        Number of observations / records.

    p : int
        Number of explanatory variables (columns of D).

    density : float, default=0.8
        Target proportion of nonzero entries in D.

    seed : int or None, default=None
        Random seed for reproducibility.

    Returns
    -------
    D : ndarray, shape (m, p)
        Design matrix.

    b : ndarray, shape (m,)
        Dependent variable.
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # check input values
    if m <= 0:
        raise ValueError("m must be positive")

    if p <= 0:
        raise ValueError("p must be positive")

    if not 0.0 <= density <= 1.0:
        raise ValueError("density must lie between 0 and 1")

    # initialize D as an all-zero matrix
    D = np.zeros((m, p), dtype=float)

    # calculate the total number of entries
    n_total = m * p

    # calculate the number of nonzero entries
    n_nonzero = int(round(density * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # fill the selected positions with values sampled from U[-1, 1]
    D.flat[nonzero_indices] = rng.uniform(-1.0, 1.0, size=n_nonzero)

    # generate the dependent variable from U[-1, 1]
    b = rng.uniform(-1.0, 1.0, size=m)

    return D, b


def build_quantile_lp(
    D: np.ndarray,
    b: np.ndarray,
    tau: float = 0.2,
):
    """
    Build the LP formulation of a quantile regression instance.

    Parameters
    ----------
    D : ndarray, shape (m, p)
        Design matrix

    b : ndarray, shape (m,)
        Dependent variable

    tau : float, default=0.2
        Quantile level

    Returns
    -------
    A : sparse matrix, shape (m, p + 2m)
        Equality constraint matrix

    c : ndarray, shape (p + 2m,)
        Objective coefficient vector

    bounds : list
        Variable bounds for beta, u_plus, and u_minus
    """

    # get the dimensions of D
    m, p = D.shape

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError("b must have shape (m,)")

    # check the quantile level
    if not 0.0 < tau < 1.0:
        raise ValueError("tau must lie strictly between 0 and 1")

    # convert D to sparse format
    D_sparse = sparse.csr_matrix(D)

    # construct a sparse identity matrix
    I = sparse.eye(m, format="csr")

    # construct A = [D | I | -I]
    A = sparse.hstack((D_sparse, I, -I), format="csr")

    # construct the objective coefficient vector
    c = np.concatenate(
        (np.zeros(p), tau * np.ones(m), (1.0 - tau) * np.ones(m))
    )

    return A, c


def generate_projection_matrix(
    A,
    k: int,
    seed: int | None = None,
):
    """
    Generate a sparse random projection matrix T.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Constraint matrix

    k : int
        Projected dimension

    seed : int or None, default=None
        Random seed for reproducibility

    Returns
    -------
    T : sparse matrix, shape (k, m)
        Sparse random projection matrix
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # get the number of original constraints
    m = A.shape[0]

    # check the projected dimension
    if not 0 < k < m:
        raise ValueError("k must lie strictly between 0 and m")

    # calculate the density of A
    density_A = A.nnz / (A.shape[0] * A.shape[1])

    # set the density of T
    sigma = density_A / 2.0

    # calculate the total number of entries in T
    n_total = k * m

    # calculate the number of nonzero entries in T
    n_nonzero = int(round(sigma * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # convert the flat indices to row and column indices
    rows = nonzero_indices // m
    cols = nonzero_indices % m

    # calculate the standard deviation of nonzero entries
    std = 1.0 / np.sqrt(k * sigma)

    # sample the nonzero entries from the Gaussian distribution
    values = rng.normal(loc=0.0, scale=std, size=n_nonzero)

    # construct the sparse projection matrix
    T = sparse.csr_matrix((values, (rows, cols)), shape=(k, m))

    return T


def project_lp(
    A,
    b: np.ndarray,
    T,
):
    """
    Project the equality constraints of an LP.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Original constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    T : sparse matrix, shape (k, m)
        Random projection matrix

    Returns
    -------
    A_projected : sparse matrix, shape (k, n)
        Projected constraint matrix

    b_projected : ndarray, shape (k,)
        Projected right-hand side vector
    """

    # check the dimensions of A and b
    if b.shape != (A.shape[0],):
        raise ValueError("b must have shape (m,)")

    # check the dimensions of T and A
    if T.shape[1] != A.shape[0]:
        raise ValueError("T and A have incompatible dimensions")

    # project the constraint matrix
    A_projected = T @ A

    # project the right-hand side vector
    b_projected = T @ b

    return A_projected, b_projected


def build_quantile_bounds(
    m: int,
    p: int,
):
    """
    Build the variable bounds for the quantile regression LP.

    Parameters
    ----------
    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    lb : ndarray, shape (p + 2m,)
        Lower bounds

    ub : ndarray, shape (p + 2m,)
        Upper bounds
    """

    # construct the lower bounds
    lb = np.concatenate((np.full(p, -np.inf), np.zeros(m), np.zeros(m)))

    # construct the upper bounds
    ub = np.full(p + 2 * m, GRB.INFINITY)

    return lb, ub


def solve_projection_system(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    x_check: np.ndarray,
):
    """
    Approximate the projection of x0 onto A x = b using LSMR.
    """

    # solve A x_hat = b + A x0
    x_hat = lsmr(A, b + A @ x0)[0]

    # construct a second point approximately satisfying A x = b
    x_prime = x_hat - x0

    # construct the direction from x_prime to x_check
    direction = x_check - x_prime

    # handle the degenerate case
    if direction @ direction == 0.0:
        return x_prime

    # calculate the projection coefficient
    lambda_value = (direction @ (x0 - x_prime)) / (direction @ direction)

    # project x0 onto the line through x_prime and x_check
    x_dagger = x_prime + lambda_value * direction

    return x_dagger


def bound_error_sum(
    x: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Calculate the sum of bound violations used in the authors' public code.
    """

    # calculate lower-bound violations
    lower_error = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_error = np.maximum(x - ub, 0.0)

    # return the sum of all bound violations
    return float(np.sum(lower_error) + np.sum(upper_error))


def retrieval_feasibility(
    x,
    A,
    b,
    lb,
    ub,
    tol=0.01,
):
    """
    Check feasibility using the error measure and tolerance
    used in the authors' public retrieval code.
    """

    # calculate the L2 equality residual
    equality_error = float(np.linalg.norm(A @ x - b))

    # calculate the sum of all bound violations
    bound_error = bound_error_sum(
        x,
        lb,
        ub,
    )

    # apply the authors' combined error criterion
    return bound_error + equality_error <= 2.0 * tol


def retrieval_start(
    x_projected: np.ndarray,
    A,
    b: np.ndarray,
):
    """
    Apply the LSMR retrieval step used before the authors' APM code.
    """

    # calculate the original equality residual of the projected LP solution
    residual = b - A @ x_projected

    # approximately solve A x_part = b - A x_projected
    x_part = lsmr(A, residual)[0]

    # construct the starting point used by the authors' APM code
    x_start = x_projected + x_part

    return x_start


def evaluate_retrieval(
    x: np.ndarray,
    A,
    b: np.ndarray,
    c: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Evaluate a retrieved solution.
    """

    # calculate the objective value
    objective_value = c @ x

    # calculate the equality constraint residual
    equality_residual = np.linalg.norm(A @ x - b)

    # calculate lower-bound violations
    lower_violation = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_violation = np.zeros_like(x)

    finite_upper = np.isfinite(ub)

    upper_violation[finite_upper] = np.maximum(
        x[finite_upper] - ub[finite_upper], 0.0
    )

    # calculate the maximum bound violation
    bound_violation = max(np.max(lower_violation), np.max(upper_violation))

    return (objective_value, equality_residual, bound_violation)


def evaluate_errors(
    x: np.ndarray,
    A,
    b: np.ndarray,
    m: int,
    p: int,
):
    """
    Calculate the average feasibility errors reported in Table 5.

    Parameters
    ----------
    x : ndarray, shape (p + 2m,)
        Retrieved quantile regression solution

    A : sparse matrix, shape (m, p + 2m)
        Original equality constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    average_inequality_error : float
        Average nonnegativity violation of u_plus and u_minus

    average_equality_error : float
        Average absolute violation of A x = b
    """

    # check the dimension of the retrieved solution
    if x.shape != (p + 2 * m,):
        raise ValueError(f"x must have shape {(p + 2 * m,)}")

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError(f"b must have shape {(m,)}")

    # extract the positive residual variables
    u_plus = x[p : p + m]

    # extract the negative residual variables
    u_minus = x[p + m : p + 2 * m]

    # calculate violations of u_plus >= 0
    u_plus_violation = np.maximum(-u_plus, 0.0)

    # calculate violations of u_minus >= 0
    u_minus_violation = np.maximum(-u_minus, 0.0)

    # calculate the average inequality error
    average_inequality_error = (
        np.sum(u_plus_violation) + np.sum(u_minus_violation)
    ) / m

    # calculate the original equality residual vector
    equality_error = A @ x - b

    # calculate the average equality error
    average_equality_error = np.mean(np.abs(equality_error))

    return (average_inequality_error, average_equality_error)


def retrieve_dykstra(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
):
    """
    Apply the Dykstra APM logic from the authors' public code.
    """

    # first apply the authors' LSMR retrieval step
    x = retrieval_start(x0, A, b)

    # calculate one approximate solution of A x = b
    x_check = lsmr(A, b)[0]

    # initialize the two Dykstra correction vectors
    p = np.zeros_like(x)
    q = np.zeros_like(x)

    # initialize the bound-projected point
    y = np.zeros_like(x)

    # use the authors' default status until another condition is met
    status = "iteration limit"

    for iteration in range(1, max_iter + 1):
        # store the point at the beginning of the iteration
        x_previous = x.copy()

        # calculate the bound-error sum before the bound projection
        bound_error = bound_error_sum(x, lb, ub)

        # the authors stop immediately if there is no material bound error
        if bound_error <= delta:
            status = "no bound error"
            return x_previous, iteration, status

        # project x + p onto the bound set
        y = np.clip(x + p, lb, ub)

        # update the first Dykstra correction vector
        p = x + p - y

        # calculate the equality residual after the bound projection
        equality_error = float(np.linalg.norm(A @ y - b))

        # project y + q onto A x = b when needed
        if equality_error > delta:
            affine_input = y + q

            x = solve_projection_system(
                affine_input,
                A,
                b,
                x_check,
            )

            # update the second Dykstra correction vector
            q = y + q - x

        # calculate the authors' no-progress measure
        progress_solution = float(np.linalg.norm(x - x_previous))

        # stop when there is no sufficient progress
        if progress_solution <= delta:
            status = "no sufficient progress"
            return x, iteration, status

        # apply the authors' combined error stopping rule
        if bound_error + equality_error <= 2.0 * delta:
            status = "small total error"
            return x, iteration, status

    return x, max_iter, status

In [ ]:
def timed_solve(A, b, c, lb, ub, method, crossover=-1, model_name="qr"):
    """Time construction + optimize + extraction; dispose outside that timing.

    optimize_wall_s includes Gurobi's pending model update, presolve, algorithm,
    and crossover when enabled. runtime_s is Gurobi Runtime, also wall time.
    It is not CPU time or a separate crossover measurement.
    """
    start = time.perf_counter()
    model = gp.Model(model_name)
    try:
        model.Params.OutputFlag = OUTPUT_FLAG
        model.Params.Method = method
        model.Params.Crossover = crossover
        model.Params.Threads = THREADS
        model.Params.Seed = SOLVER_SEED
        x = model.addMVar(shape=A.shape[1], lb=lb, ub=ub, name="x")
        model.setObjective(c @ x, GRB.MINIMIZE)
        model.addConstr(A @ x == b, name="equality_constraints")
        before_optimize = time.perf_counter()
        model.optimize()
        after_optimize = time.perf_counter()
        status = model.Status
        solution = x.X.copy() if model.SolCount else None
        objective = model.ObjVal if model.SolCount else np.nan
        total_wall_s = time.perf_counter() - start
        info = dict(
            status="OPTIMAL" if status == GRB.OPTIMAL else f"status {status}",
            status_code=status,
            objective=objective,
            setup_wall_s=before_optimize - start,
            optimize_wall_s=after_optimize - before_optimize,
            total_wall_s=total_wall_s,
            runtime_s=model.Runtime,
            work=model.Work,
            simplex_iterations=model.IterCount,
            barrier_iterations=model.BarIterCount,
            constraint_violation=model.ConstrVio if solution is not None else np.nan,
            bound_violation=model.BoundVio if solution is not None else np.nan,
        )
        return solution, info
    finally:
        model.dispose()


# Initialize the default license/environment once outside all measured solves.
# This excludes one-time license startup fairly from every algorithm and the
# projected solve. No LP is optimized here and no basis is reused.
startup_begin = time.perf_counter()
with gp.Model("initialize_environment") as startup_model:
    solver_defaults = {
        name: getattr(startup_model.Params, name)
        for name in ("Presolve", "FeasibilityTol", "OptimalityTol", "BarConvTol")
    }
startup_wall_s = time.perf_counter() - startup_begin
environment = dict(
    timestamp=datetime.now().astimezone().isoformat(),
    python=sys.version.split()[0], gurobi=".".join(map(str, gp.gurobi.version())),
    numpy=np.__version__, scipy=scipy.__version__,
    os=platform.platform(), cpu=platform.processor(), logical_cpus=os.cpu_count(),
    threads_parameter=THREADS, solver_seed=SOLVER_SEED,
    output_flag=OUTPUT_FLAG, excluded_license_startup_s=startup_wall_s,
    **solver_defaults,
)
# Full environment details remain in `environment`.


In [ ]:
# Fixed numerator: execute this cell once before all original algorithm solves.
D, b = generate_quantile_instance(M, P, DENSITY, DATA_SEED)
A, c = build_quantile_lp(D, b, TAU)
lb, ub = build_quantile_bounds(M, P)
k = int(round((1.0 / EPSILON) ** 2) * np.log(float(A.shape[1])))
if not 0 < k < M:
    raise ValueError(f"Invalid projected dimension: k={k}, m={M}")

start = time.perf_counter()
T = generate_projection_matrix(A, k, PROJECTION_SEED)
time_T = time.perf_counter() - start
start = time.perf_counter()
A_projected, b_projected = project_lp(A, b, T)
time_projection = time.perf_counter() - start

x_tilde, projected_info = timed_solve(
    A_projected, b_projected, c, lb, ub, method=1,
    crossover=-1, model_name="fixed_projected_dual_simplex",
)
if projected_info["status_code"] != GRB.OPTIMAL:
    raise RuntimeError(f"Fixed projected solve failed: {projected_info['status']}")
start = time.perf_counter()
x_dykstra, dykstra_iterations, dykstra_status = retrieve_dykstra(
    x_tilde, A, b, lb, ub, max_iter=MAX_ITER, delta=DELTA,
)
time_retrieval = time.perf_counter() - start
fixed_pipeline_s = time_T + time_projection + projected_info["total_wall_s"] + time_retrieval

retrieved_objective, retrieved_l2, retrieved_max_bound = evaluate_retrieval(
    x_dykstra, A, b, c, lb, ub,
)
avgin, avgeq = evaluate_errors(x_dykstra, A, b, M, P)
combined_error = bound_error_sum(x_dykstra, lb, ub) + retrieved_l2
retrieved_feasible = retrieval_feasibility(x_dykstra, A, b, lb, ub, tol=DELTA)


In [ ]:
# Fresh, sequential original solves. Original solutions never enter retrieval.
rows = []
for repeat in range(REPEATS):
    shift = repeat % len(ALGORITHMS)
    order = ALGORITHMS[shift:] + ALGORITHMS[:shift]
    for position, (name, method, crossover) in enumerate(order, start=1):
        print(f"Solving {name} ({repeat + 1}/{REPEATS})...", flush=True)
        try:
            solution, info = timed_solve(
                A, b, c, lb, ub, method, crossover,
                model_name=f"original_r{repeat + 1}_method{method}_cross{crossover}",
            )
            # Quality checks excluded from benchmark solve time.
            if solution is not None:
                _, eq_l2, max_bound = evaluate_retrieval(solution, A, b, c, lb, ub)
            else:
                eq_l2, max_bound = np.nan, np.nan
            valid = info["status_code"] == GRB.OPTIMAL
            info.update(
                equality_l2=eq_l2, max_bound_violation=max_bound,
                pipeline_over_original=fixed_pipeline_s / info["total_wall_s"] if valid else np.nan,
                retrieved_over_original_objective=(
                    retrieved_objective / info["objective"]
                    if valid and retrieved_feasible and abs(info["objective"]) > 1e-12
                    else np.nan
                ),
            )
        except gp.GurobiError as error:
            info = {"status": f"Gurobi error {error.errno}: {error.message}"}
        rows.append(dict(algorithm=name, method=method, crossover=crossover,
                         repeat=repeat + 1, order_position=position, **info))
results = pd.DataFrame(rows)


In [ ]:
# Essential results; full per-run measurements remain in `results`.
print(f"Gurobi {environment['gurobi']} | Threads={THREADS} (0=automatic) | Runs per algorithm={REPEATS}")
print(f"CPU: {environment['cpu']} | Logical CPUs: {environment['logical_cpus']}")
print(f"m={M}, p={P}, epsilon={EPSILON}, k={k} | Data seed={DATA_SEED}, projection seed={PROJECTION_SEED}")
print("Wall seconds include model setup; shared data generation and license startup excluded.")

print("\n===== Fixed pipeline wall-clock time (seconds; measured once) =====")
print(f"Generate T:           {time_T:.3f}")
print(f"Compute TA and Tb:    {time_projection:.3f}")
print(f"Solve TLP:            {projected_info['total_wall_s']:.3f}")
print(f"Dykstra retrieval:    {time_retrieval:.3f}")
print(f"RP + TLP + Dykstra:   {fixed_pipeline_s:.3f}")

print("\n===== Fixed projected / Dykstra solution =====")
print(f"Projected objective:       {projected_info['objective']:.6f}")
print(f"Dykstra objective:         {retrieved_objective:.6f}")
print(f"Equality residual (L2):    {retrieved_l2:.3e}")
print(f"Maximum bound violation:   {retrieved_max_bound:.3e}")
print(f"Average inequality error:  {avgin:.3e}")
print(f"Average equality error:    {avgeq:.3e}")
print(f"Combined error:            {combined_error:.6f} (required <= {2*DELTA:.3f})")
print(f"Feasible at tolerance:     {retrieved_feasible}")
print(f"Iterations / stop:         {dykstra_iterations} / {dykstra_status}")
if not retrieved_feasible:
    print("Retrieved point fails the reference feasibility test; ratios compare timings only.")

summary_rows = []
for name, method, crossover in ALGORITHMS:
    group = results[results.algorithm == name]
    ok = group[group.status == "OPTIMAL"]
    print(f"\n===== {name} =====")
    print(f"Status:                    {'OPTIMAL' if len(ok) == REPEATS else 'INCOMPLETE'} ({len(ok)}/{REPEATS} runs)")
    for status in group[group.status != "OPTIMAL"].status:
        print(f"Solve issue:               {status}")
    if not len(ok):
        continue
    if len(ok) < REPEATS:
        print("Mean uses optimal runs only; the five-run comparison is incomplete.")
    original_time = float(ok.total_wall_s.mean())
    ratio = fixed_pipeline_s / original_time
    objective = float(ok.objective.mean())
    print(f"Original LP mean time:     {original_time:.3f} s")
    print(f"Time range:               {ok.total_wall_s.min():.3f} - {ok.total_wall_s.max():.3f} s")
    print(f"Mean objective value:      {objective:.6f}")
    print(f"Worst equality L2:         {ok.equality_l2.max():.3e}")
    print(f"Worst bound violation:     {ok.max_bound_violation.max():.3e}")
    print(f"RP / original time ratio:  {ratio:.3f}")
    print(f"Direct / Dykstra pipeline: {original_time / fixed_pipeline_s:.3f}x")
    summary_rows.append(dict(algorithm=name, mean_original_s=original_time,
                             objective=objective, ratio=ratio))
summary = pd.DataFrame(summary_rows)

optimal = results[results.status == "OPTIMAL"]
if len(optimal):
    agrees = np.allclose(optimal.objective, optimal.objective.iloc[0], rtol=1e-6, atol=1e-7)
    print(f"\nOriginal objectives agree across optimal runs: {agrees}")
print(f"Table 5 timing ratio: {PAPER_RATIO:.2f} | Previous observation: {OBSERVED_RATIO:.3f}")
print("Ratio = fixed RP pipeline / mean original solve time. Slower direct solves lower this ratio.")
print("Table 5 uses different solvers, hardware, and timing scope; this comparison isolates Gurobi algorithms.")


In [ ]:
# Compact table: effect of the original Gurobi algorithm on timing.
from IPython.display import display

if "results" not in globals():
    raise RuntimeError("Run the original-LP comparison cell first; this cell reuses its results.")

automatic_runs = results[(results.algorithm == "Automatic") & (results.status == "OPTIMAL")]
automatic_mean_s = automatic_runs.total_wall_s.mean() if len(automatic_runs) else np.nan
comparison_rows = []
for name, method, crossover in ALGORITHMS:
    runs = results[results.algorithm == name]
    optimal_runs = runs[runs.status == "OPTIMAL"]
    mean_total_s = optimal_runs.total_wall_s.mean() if len(optimal_runs) else np.nan
    comparison_rows.append({
        "Gurobi algorithm": name,
        "Optimal runs": f"{len(optimal_runs)}/{REPEATS}",
        "Mean total (s)": mean_total_s,
        "SD total (s)": optimal_runs.total_wall_s.std(ddof=1) if len(optimal_runs) > 1 else np.nan,
        "Mean optimize (s)": optimal_runs.optimize_wall_s.mean() if len(optimal_runs) else np.nan,
        "Time / Automatic": mean_total_s / automatic_mean_s if automatic_mean_s > 0 else np.nan,
        "RP / original": fixed_pipeline_s / mean_total_s if mean_total_s > 0 else np.nan,
        "Mean objective": optimal_runs.objective.mean() if len(optimal_runs) else np.nan,
    })
algorithm_comparison_table = pd.DataFrame(comparison_rows)

print("===== Original-LP algorithm comparison (five-run means) =====")
print(f"Fixed RP + TLP + Dykstra time: {fixed_pipeline_s:.3f} s | Table 5 ratio: {PAPER_RATIO:.2f}")
display(algorithm_comparison_table.style.hide(axis="index").format({
    "Mean total (s)": "{:.3f}", "SD total (s)": "{:.3f}",
    "Mean optimize (s)": "{:.3f}", "Time / Automatic": "{:.2f}x",
    "RP / original": "{:.3f}", "Mean objective": "{:.6f}",
}, na_rep="?"))
print("Mean total includes model setup, optimization, and solution extraction; SD shows timing variation.")
print("Mean optimize measures the optimize() call, including pending updates, presolve, and crossover when enabled.")
print("Time / Automatic > 1 means slower than Automatic. RP / original uses the fixed numerator divided by mean total time.")
valid_runs = results[results.status == "OPTIMAL"]
if len(valid_runs):
    objective_agreement = np.allclose(valid_runs.objective, valid_runs.objective.iloc[0], rtol=1e-6, atol=1e-7)
    print(f"Original objectives agree across optimal runs: {objective_agreement}")
if any(results[results.algorithm == name].status.eq("OPTIMAL").sum() != REPEATS for name, _, _ in ALGORITHMS):
    print("Incomplete comparison: means use available optimal runs only; inspect solve failures above.")
print(f"Fixed retrieved point passes the reference feasibility test: {retrieved_feasible}; ratios describe timings.")
print("This isolates algorithm choice within Gurobi on this instance and machine.")
print("To isolate solver implementation differences, compare with CPLEX on the same instance, hardware, and timing scope.")
